# Tema 6.1: Evaluación de Segmentación - Métricas IoU y coeficiente Dice

*Duración estimada: 1 hora*

## 1. La importancia de la Evaluación Matemática

En Machine Learning, si no puedes medirlo, no puedes mejorarlo. Cuando predecimos máscaras de segmentación, no basta con "ver que se ve bien". Necesitamos una métrica numérica que compare la máscara generada por nuestro modelo (Prediction) contra la máscara real dibujada por un humano experto (Ground Truth).

Las dos métricas rey en Visión Computacional son **IoU (Intersection over Union)** y **Dice Coefficient**.

## 2. IoU (Intersection over Union) / Índice Jaccard

Es la métrica más intuitiva. Se responde a la pregunta: *De todo el espacio que ocupan juntas la predicción y la realidad, ¿qué porcentaje está perfectamente sobrepuesto?*

**Fórmula:** 
$$ IoU = \frac{Area\ de\ Interseccion}{Area\ de\ Union} $$

Va de 0 (ningún solapamiento) a 1 (solapamiento perfecto). En detección de objetos, un IoU > 0.5 suele considerarse un acierto.

## 3. Coeficiente Dice (F1-Score espacial)

El coeficiente Dice penaliza de forma ligeramente diferente los falsos positivos y falsos negativos, dándole un peso doble a la intersección.

**Fórmula:**
$$ Dice = \frac{2 \times Area\ de\ Interseccion}{Area\ Total\ Predicha + Area\ Total\ Real} $$

Es muy utilizado en imagenología médica, ya que maneja mejor los desbalances (ej. un tumor muy pequeñito en una imagen muy grande).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_masks(ground_truth, prediction):
    """Función de apoyo para visualizar el solapamiento"""
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    
    ax[0].imshow(ground_truth, cmap='Blues')
    ax[0].set_title('Ground Truth (Verdad)')
    
    ax[1].imshow(prediction, cmap='Oranges')
    ax[1].set_title('Prediction (Modelo)')
    
    # Superposición: Verdad en azul, Predicción en naranja, Solapamiento en marrón oscuro
    overlap = np.zeros((*ground_truth.shape, 3))
    overlap[ground_truth] = [0, 0, 1] # Azul
    overlap[prediction] = [1, 0.5, 0] # Naranja
    overlap[np.logical_and(ground_truth, prediction)] = [0.5, 0.25, 0] # Solapamiento
    
    ax[2].imshow(overlap)
    ax[2].set_title('Superposición')
    plt.show()

In [ ]:
def compute_iou_and_dice(y_true, y_pred):
    # 1. Asegurar booleanos
    y_true = np.asarray(y_true).astype(bool)
    y_pred = np.asarray(y_pred).astype(bool)
    
    # 2. Calcular Intersección
    intersection = np.logical_and(y_true, y_pred).sum()
    
    # 3. Calcular Unión
    union = np.logical_or(y_true, y_pred).sum()
    
    # 4. Suma de áreas
    total_area = y_true.sum() + y_pred.sum()
    
    # 5. Evitar división por cero
    if union == 0:
        iou = 1.0 if total_area == 0 else 0.0
    else:
        iou = intersection / union
        
    dice = (2. * intersection) / total_area if total_area > 0 else 1.0
    
    return iou, dice

## 4. Ejercicio Práctico: Simulando Solapamientos

Vamos a crear máscaras de ejemplo y calcular sus métricas.

In [ ]:
# Crear matriz base de 100x100
gt_mask = np.zeros((100, 100), dtype=bool)
pred_mask_good = np.zeros((100, 100), dtype=bool)
pred_mask_bad = np.zeros((100, 100), dtype=bool)

# Ground Truth: Rectángulo de [30:70, 30:70]
gt_mask[30:70, 30:70] = True

# Buena predicción (Ligeramente desplazada)
pred_mask_good[35:75, 30:70] = True

# Mala predicción (Apenas se toca)
pred_mask_bad[65:95, 65:95] = True

print("--- ESCENARIO 1: Buena Predicción ---")
plot_masks(gt_mask, pred_mask_good)
iou, dice = compute_iou_and_dice(gt_mask, pred_mask_good)
print(f"IoU: {iou:.3f} | Dice: {dice:.3f}\n")

print("--- ESCENARIO 2: Mala Predicción ---")
plot_masks(gt_mask, pred_mask_bad)
iou, dice = compute_iou_and_dice(gt_mask, pred_mask_bad)
print(f"IoU: {iou:.3f} | Dice: {dice:.3f}")

## 5. Conclusión de Métricas

Notarás que el coeficiente Dice siempre es ligeramente superior al IoU cuando hay solapamiento parcial. 
- **IoU** es más estricto y penaliza mucho los errores. Es el estándar de oro.
- **Dice** se usa más para optimizar modelos de redes neuronales (como pérdida matemática) porque su derivada es más suave.